# **Cell 1: Data Preprocessing and Setup**
This cell handles unzipping data, encoding, and scaling features and targets exactly as before.


In [2]:
import os
import zipfile
import numpy as np
import pandas as pd

# Unzip data if needed
zip_path = "/content/archive.zip"
extract_path = "/content/"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

# Load Dataset
df = pd.read_csv("/content/insurance.csv")

# One-Hot Encode Categorical Features
df = pd.get_dummies(df, columns=["sex", "smoker", "region"], drop_first=True).astype(float)

# Features and Target
X = df.drop("charges", axis=1).values
y = df["charges"].values.reshape(-1, 1)

# Feature Scaling
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X = (X - X_mean) / X_std

y_mean, y_std = y.mean(), y.std()
y = (y - y_mean) / y_std

# Train-Test Split
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

X_train, y_train = X[indices[:split]], y[indices[:split]]
X_test, y_test = X[indices[split:]], y[indices[split:]]

# Cell 2: MLP Class Definition (NumPy Backpropagation)
This model has an Input Layer, a Hidden Layer using a ReLU activation function, and a Linear Output Layer.


In [15]:

class MLPRegressor:
    def __init__(self, hidden_dim=64, learning_rate=0.005, epochs=1500, batch_size=32):
        self.hidden_dim = hidden_dim
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        # Weights and biases initialized during fit()
        self.W1, self.b1 = None, None
        self.W2, self.b2 = None, None

    def _relu(self, Z):
        return np.maximum(0, Z)

    def _relu_derivative(self, Z):
        return (Z > 0).astype(float)

    def fit(self, X, y, X_test, y_test):
        n_samples, n_features = X.shape

        # He (Kaiming) Initialization for hidden layer, Xavier for output layer
        np.random.seed(42)
        self.W1 = np.random.randn(n_features, self.hidden_dim) * np.sqrt(2.0 / n_features)
        self.b1 = np.zeros((1, self.hidden_dim))
        self.W2 = np.random.randn(self.hidden_dim, 1) * np.sqrt(1.0 / self.hidden_dim)
        self.b2 = np.zeros((1, 1))

        for epoch in range(self.epochs):
            # Shuffle every epoch
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, n_samples, self.batch_size):
                X_b = X_shuffled[i : i + self.batch_size]
                y_b = y_shuffled[i : i + self.batch_size]
                m = X_b.shape[0]

                # --- 1. Forward Pass ---
                Z1 = np.dot(X_b, self.W1) + self.b1
                A1 = self._relu(Z1)
                Z2 = np.dot(A1, self.W2) + self.b2
                y_pred = Z2  # Linear activation for output layer

                # --- 2. Backward Pass (Chain Rule) ---
                # Loss derivative with respect to output prediction (MSE)
                dZ2 = (2 / m) * (y_pred - y_b)

                # Gradients for layer 2
                dW2 = np.dot(A1.T, dZ2)
                db2 = np.sum(dZ2, axis=0, keepdims=True)

                # Backprop through hidden layer
                dA1 = np.dot(dZ2, self.W2.T)
                dZ1 = dA1 * self._relu_derivative(Z1)

                # Gradients for layer 1
                dW1 = np.dot(X_b.T, dZ1)
                db1 = np.sum(dZ1, axis=0, keepdims=True)

                # --- 3. Parameter Updates ---
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2

            # Print loss every 200 epochs
            if epoch % 500 == 0:
                total_pred = self.predict(X)
                loss = np.mean((total_pred - y) ** 2)
                test_pred = self.predict(X_test)
                test_loss = np.mean((test_pred - y_test) ** 2)
                print(f"Epoch {epoch:04d}, Training Loss = {loss:.6f}, Test loss = {test_loss:.6f}")

    def predict(self, X):
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = self._relu(Z1)
        Z2 = np.dot(A1, self.W2) + self.b2
        return Z2

# Cell 3: Training and Evaluation

In [23]:

# Instantiate and train MLP
mlp_model = MLPRegressor(hidden_dim=256, learning_rate=0.005, epochs=501, batch_size=64)
mlp_model.fit(X_train, y_train, X_test, y_test)

# Evaluate on test set
y_pred_scaled = mlp_model.predict(X_test)

# Scale back to normal currency values
y_pred = y_pred_scaled * y_std + y_mean
y_actual = y_test * y_std + y_mean

# Calculate Metrics
mse = np.mean((y_pred - y_actual) ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum((y_actual - y_pred) ** 2)
ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("\n--- MLP Results ---")
print(f"RMSE: ${rmse:.2f}")
print(f"R² Score: {r2:.4f}")



Epoch 0000, Training Loss = 0.296357, Test loss = 0.332532
Epoch 0500, Training Loss = 0.123499, Test loss = 0.180026

--- MLP Results ---
RMSE: $5136.30
R² Score: 0.8310


# Cell 4: Predicting New Individual


In [24]:
new_person = {
    "age": 35,
    "bmi": 28.5,
    "children": 2,
    "sex_male": 1,
    "smoker_yes": 0,
    "region_northwest": 0,
    "region_southeast": 1,
    "region_southwest": 0
}

new_df = pd.DataFrame([new_person])
new_X = (new_df.values - X_mean) / X_std

# Perform prediction using trained weights
pred_scaled = mlp_model.predict(new_X)
pred_charge = pred_scaled * y_std + y_mean

print(f"Predicted MLP Insurance Charge: ${pred_charge[0][0]:.2f}")

Predicted MLP Insurance Charge: $7897.77
